In [12]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
#from langchain_community.vectorstores import Chroma
from langchain_chroma import Chroma
import os

load_dotenv(override=True)
DOCS_DIR    = "./docs" 
DOCS_DIR    = "./docs"        # carpeta con tus PDFs y TXTs
CHROMA_DIR  = "./chroma_db"   # persistencia del vector store
EMBED_MODEL = "text-embedding-3-small"
TOP_K       = 4               # chunks a recuperar por consulta

In [3]:
def cargar_documentos(ruta: str = DOCS_DIR):
    """Carga todos los PDFs de una carpeta de forma recursiva."""

    documentos = []
    loader = DirectoryLoader(
        ruta,
        glob="**/*.pdf",
        loader_cls=PyPDFLoader
    )
    pdfs = loader.load()
    documentos.extend(pdfs)
    print(f"✅ {len(pdfs)} páginas PDF cargadas")

    txt_loader = DirectoryLoader(
        ruta,
        glob="**/*.txt",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )
    txts = txt_loader.load()
    documentos.extend(txts)
    print(f"✅ {len(txts)} archivos TXT cargados")
    
    print(f"📄 Total: {len(documentos)} documentos")

    return documentos

In [4]:
doc= cargar_documentos("./docs")
doc[0].metadata

✅ 4 páginas PDF cargadas
✅ 1 archivos TXT cargados
📄 Total: 5 documentos


{'producer': 'Microsoft® Word 2019',
 'creator': 'Microsoft® Word 2019',
 'creationdate': '2026-02-12T12:51:52-03:00',
 'author': 'SION',
 'moddate': '2026-02-12T12:51:52-03:00',
 'source': 'docs\\CV_JuanCruzJurado.pdf',
 'total_pages': 4,
 'page': 0,
 'page_label': '1'}

In [5]:
doc[0].page_content[:500]

'Juan Cruz Jurado Auzza \nInformación de contacto \n \nPaís: Argentina \nTeléfono:  +54 9 351 321 7599 \nEmail:  juradojuancruz@gmail.com \nLinkedIn: linkedin.com/in/juan-cruz-jurado-auzza \nGitHub: github.com/jcjurado \nPerfil Profesional \nSenior Data Engineer / BI Consultant con más de 10 años de experiencia en retail, energía y \nsector público. Especializado en Python, SQL, Apache Airflow y BigQuery, con foco en diseño de \npipelines ETL/ELT y migraciones de DataWarehouse desde Oracle hacia Google Clou'

In [6]:
def dividir_documentos(documentos):
    """Divide los documentos en chunks con overlap para no perder contexto."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    chunks = splitter.split_documents(documentos)
    print(f"✅ {len(chunks)} chunks generados")
    return chunks

In [7]:
chunk = dividir_documentos(doc)

✅ 13 chunks generados


In [8]:
chunk[0].page_content

'Juan Cruz Jurado Auzza \nInformación de contacto \n \nPaís: Argentina \nTeléfono:  +54 9 351 321 7599 \nEmail:  juradojuancruz@gmail.com \nLinkedIn: linkedin.com/in/juan-cruz-jurado-auzza \nGitHub: github.com/jcjurado \nPerfil Profesional \nSenior Data Engineer / BI Consultant con más de 10 años de experiencia en retail, energía y \nsector público. Especializado en Python, SQL, Apache Airflow y BigQuery, con foco en diseño de'

In [9]:
def crear_vectorstore(chunks, persist_directory: str = CHROMA_DIR):
    """Indexa los chunks y persiste el vector store en disco."""
    embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_directory
    )
    print(f"✅ Vector store creado en '{persist_directory}'")
    return vectorstore

def cargar_vectorstore(persist_directory: str = CHROMA_DIR):
    """Carga un vector store ya existente desde disco."""
    embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
    return Chroma(
        persist_directory=persist_directory,
        embedding_function=embeddings
    )

In [10]:
class RAG:
    """
    Interfaz simple para búsqueda semántica sobre los documentos del perfil.

    Uso típico en chat.py:
        rag = RAG()
        contexto = rag.buscar("¿Qué experiencia tiene en Python?")
    """

    def __init__(self, docs_dir: str = DOCS_DIR, chroma_dir: str = CHROMA_DIR):
        if os.path.exists(chroma_dir):
            print("📂 Cargando vector store existente...")
            self.vectorstore = cargar_vectorstore(chroma_dir)
        else:
            print("📂 Indexando documentos por primera vez...")
            documentos = cargar_documentos(docs_dir)
            chunks     = dividir_documentos(documentos)
            self.vectorstore = crear_vectorstore(chunks, chroma_dir)

        self.retriever = self.vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={"k": TOP_K}
        )
        print("🤖 RAG listo\n")

    def buscar(self, query: str) -> str:
        """
        Devuelve los chunks más relevantes como texto plano,
        listos para inyectar en el system prompt.
        """
        docs = self.retriever.invoke(query)
        if not docs:
            return ""
        fragmentos = []
        for i, doc in enumerate(docs, 1):
            fuente = doc.metadata.get("source", "desconocida")
            pagina = doc.metadata.get("page", "?")
            fragmentos.append(
                f"[Fragmento {i} | {fuente} p.{pagina}]\n{doc.page_content}"
            )
        return "\n\n".join(fragmentos)

    def reindexar(self, docs_dir: str = DOCS_DIR, chroma_dir: str = CHROMA_DIR):
        """Fuerza la re-indexación aunque ya exista el vector store."""
        documentos = cargar_documentos(docs_dir)
        chunks     = dividir_documentos(documentos)
        self.vectorstore = crear_vectorstore(chunks, chroma_dir)
        self.retriever   = self.vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={"k": TOP_K}
        )
        print("🔄 Re-indexación completada")

In [15]:
message=("cual es experiencia?")
rag = RAG()
contexto = rag.buscar(message)

📂 Cargando vector store existente...
🤖 RAG listo



In [16]:
contexto

''